# EDA: Ventas MercadoLibre — Bronze Layer
**Tabla:** `iniciacion_deportiva.bronze.ventas_raw`  
**Objetivo:** Entender la estructura, calidad y distribución de los datos para planificar la capa Silver.

> **Motto del Data Engineer:** *"No podés limpiar lo que no entendés"*

---
## Contexto del dataset
- Ventas reales del negocio familiar desde MercadoLibre API
- Cargadas mes a mes: 2025-01-01 → 2026-03-04 (~9600 registros)
- Todo en **STRING** en Bronze — castear es trabajo de Silver
- `_rescued_data`: Auto Loader la llena si llega un campo fuera de schema
- **Nota de negocio:** Un cliente puede tener múltiples `id_venta` con la misma fecha/hora porque MercadoLibre devuelve una orden por producto. En Gold los agruparemos para reconstruir pedidos completos.

## Paso 1: Estructura y Exploración Inicial
*¿Qué tenemos?*

In [ ]:
%sql
-- 1.1: ¿Cuántos registros tenemos?
SELECT COUNT(*) AS total_registros
FROM iniciacion_deportiva.bronze.ventas_raw;

In [ ]:
%sql
-- 1.2: Ver las primeras filas sin columnas de metadata
SELECT
    id_venta,
    fecha,
    id_producto,
    titulo_producto,
    cantidad,
    categoria_id,
    precio_unitario,
    comision,
    id_cliente,
    cliente_nickname,
    estado
FROM iniciacion_deportiva.bronze.ventas_raw
LIMIT 20;

In [ ]:
%sql
-- 1.3: Cardinalidad — ¿cuántos valores únicos tiene cada columna clave?
SELECT
    COUNT(DISTINCT id_venta)        AS ventas_unicas,
    COUNT(DISTINCT id_producto)     AS productos_unicos,
    COUNT(DISTINCT id_cliente)      AS clientes_unicos,
    COUNT(DISTINCT categoria_id)    AS categorias_unicas,
    COUNT(DISTINCT estado)          AS estados_unicos,
    COUNT(DISTINCT titulo_producto) AS titulos_unicos
FROM iniciacion_deportiva.bronze.ventas_raw;

## Paso 2: Análisis de Nulos
*¿Qué tan completos están los datos?*

In [ ]:
%sql
-- 2.1: Conteo absoluto de nulos por columna
WITH total AS (
    SELECT COUNT(*) AS n
    FROM iniciacion_deportiva.bronze.ventas_raw
),
nulos AS (
    SELECT
        COUNT(*) - COUNT(id_venta)         AS nulos_id_venta,
        COUNT(*) - COUNT(fecha)            AS nulos_fecha,
        COUNT(*) - COUNT(id_producto)      AS nulos_id_producto,
        COUNT(*) - COUNT(titulo_producto)  AS nulos_titulo,
        COUNT(*) - COUNT(cantidad)         AS nulos_cantidad,
        COUNT(*) - COUNT(categoria_id)     AS nulos_categoria,
        COUNT(*) - COUNT(precio_unitario)  AS nulos_precio,
        COUNT(*) - COUNT(comision)         AS nulos_comision,
        COUNT(*) - COUNT(id_cliente)       AS nulos_id_cliente,
        COUNT(*) - COUNT(cliente_nickname) AS nulos_nickname,
        COUNT(*) - COUNT(estado)           AS nulos_estado
    FROM iniciacion_deportiva.bronze.ventas_raw
)
SELECT t.n AS total_registros, n.*
FROM total t, nulos n;

In [ ]:
%sql
-- 2.2: Porcentaje de nulos en columnas clave
WITH totales AS (
    SELECT COUNT(*) AS total FROM iniciacion_deportiva.bronze.ventas_raw
)
SELECT
    ROUND((t.total - COUNT(id_venta))        * 100.0 / t.total, 2) AS pct_nulos_id_venta,
    ROUND((t.total - COUNT(fecha))           * 100.0 / t.total, 2) AS pct_nulos_fecha,
    ROUND((t.total - COUNT(precio_unitario)) * 100.0 / t.total, 2) AS pct_nulos_precio,
    ROUND((t.total - COUNT(comision))        * 100.0 / t.total, 2) AS pct_nulos_comision,
    ROUND((t.total - COUNT(id_cliente))      * 100.0 / t.total, 2) AS pct_nulos_id_cliente,
    ROUND((t.total - COUNT(estado))          * 100.0 / t.total, 2) AS pct_nulos_estado
FROM iniciacion_deportiva.bronze.ventas_raw
CROSS JOIN totales t
GROUP BY t.total;

In [ ]:
%sql
-- 2.3: ¿Hay _rescued_data?
-- Auto Loader la llena cuando un campo llega en formato inesperado (schema mismatch)
-- Si hay valores acá, algo en la API cambió y Bronze lo capturó igual
SELECT
    COUNT(*)                                            AS total_registros,
    COUNT(_rescued_data)                                AS con_rescued_data,
    ROUND(COUNT(_rescued_data) * 100.0 / COUNT(*), 2)  AS pct_rescued
FROM iniciacion_deportiva.bronze.ventas_raw;

## Paso 3: Cardinalidad y Distribución de Categóricos
*¿Qué valores tienen las columnas de texto?*

In [ ]:
%sql
-- 3.1: Distribución de estados de venta
-- Clave para entender tasa de cancelación — dato central para Gold
SELECT
    estado,
    COUNT(*)                                           AS cantidad,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS porcentaje
FROM iniciacion_deportiva.bronze.ventas_raw
GROUP BY estado
ORDER BY cantidad DESC;

In [ ]:
%sql
-- 3.2: Top 20 productos más vendidos (solo órdenes pagadas)
SELECT
    id_producto,
    titulo_producto,
    COUNT(*)                   AS ordenes,
    SUM(CAST(cantidad AS INT)) AS unidades_totales
FROM iniciacion_deportiva.bronze.ventas_raw
WHERE estado = 'paid'
GROUP BY id_producto, titulo_producto
ORDER BY ordenes DESC
LIMIT 20;

In [ ]:
%sql
-- 3.3: Distribución por categoría
SELECT
    categoria_id,
    COUNT(*)                                           AS ordenes,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS porcentaje
FROM iniciacion_deportiva.bronze.ventas_raw
GROUP BY categoria_id
ORDER BY ordenes DESC;

In [ ]:
%sql
-- 3.4: Top 20 clientes por frecuencia de compra
SELECT
    id_cliente,
    cliente_nickname,
    COUNT(*)                   AS ordenes,
    SUM(CAST(cantidad AS INT)) AS unidades_totales
FROM iniciacion_deportiva.bronze.ventas_raw
WHERE estado = 'paid'
GROUP BY id_cliente, cliente_nickname
ORDER BY ordenes DESC
LIMIT 20;

## Paso 4: Rangos y Outliers en Variables Numéricas
*¿Los números tienen sentido?*

> **Importante:** Todo viene como STRING desde Bronze. Casteamos acá solo para analizar.
> En Silver lo hacemos definitivamente con `TRY_CAST` y validaciones.

In [ ]:
%sql
-- 4.1: Estadísticas de precio_unitario
SELECT
    COUNT(*)                                                     AS registros,
    MIN(CAST(precio_unitario AS DOUBLE))                         AS precio_min,
    MAX(CAST(precio_unitario AS DOUBLE))                         AS precio_max,
    ROUND(AVG(CAST(precio_unitario AS DOUBLE)), 2)               AS precio_promedio,
    ROUND(PERCENTILE(CAST(precio_unitario AS DOUBLE), 0.5),  2) AS precio_mediana,
    ROUND(PERCENTILE(CAST(precio_unitario AS DOUBLE), 0.25), 2) AS precio_p25,
    ROUND(PERCENTILE(CAST(precio_unitario AS DOUBLE), 0.75), 2) AS precio_p75
FROM iniciacion_deportiva.bronze.ventas_raw
WHERE precio_unitario IS NOT NULL AND precio_unitario != '';

In [ ]:
%sql
-- 4.2: Estadísticas de cantidad vendida
SELECT
    COUNT(*)                                   AS registros,
    MIN(CAST(cantidad AS INT))                 AS cantidad_min,
    MAX(CAST(cantidad AS INT))                 AS cantidad_max,
    ROUND(AVG(CAST(cantidad AS DOUBLE)), 2)    AS cantidad_promedio,
    PERCENTILE(CAST(cantidad AS INT), 0.5)     AS cantidad_mediana,
    PERCENTILE(CAST(cantidad AS INT), 0.95)    AS cantidad_p95
FROM iniciacion_deportiva.bronze.ventas_raw
WHERE cantidad IS NOT NULL AND cantidad != '';

In [ ]:
%sql
-- 4.3: Outliers en precio — ¿hay valores absurdos?
WITH stats AS (
    SELECT
        PERCENTILE(CAST(precio_unitario AS DOUBLE), 0.01) AS p01,
        PERCENTILE(CAST(precio_unitario AS DOUBLE), 0.99) AS p99
    FROM iniciacion_deportiva.bronze.ventas_raw
    WHERE precio_unitario IS NOT NULL AND precio_unitario != ''
)
SELECT 'Muy bajo (< P01)' AS tipo_outlier, COUNT(*) AS cantidad
FROM iniciacion_deportiva.bronze.ventas_raw, stats
WHERE CAST(precio_unitario AS DOUBLE) < stats.p01
UNION ALL
SELECT 'Muy alto (> P99)' AS tipo_outlier, COUNT(*) AS cantidad
FROM iniciacion_deportiva.bronze.ventas_raw, stats
WHERE CAST(precio_unitario AS DOUBLE) > stats.p99;

In [ ]:
%sql
-- 4.4: ¿Hay precios en 0 o negativos? (datos claramente inválidos)
SELECT COUNT(*) AS precios_invalidos
FROM iniciacion_deportiva.bronze.ventas_raw
WHERE TRY_CAST(precio_unitario AS DOUBLE) <= 0
   OR precio_unitario IS NULL
   OR precio_unitario = '';

## Paso 5: Duplicados y Calidad General
*¿Los datos son confiables?*

> En Bronze **se esperan duplicados** — el ETL incremental corre cada 15 min con ventana de 2 horas,
> entonces cada venta puede aparecer hasta ~8 veces. Silver los elimina con `MERGE ON id_venta`.

In [ ]:
%sql
-- 5.1: ¿Cuántos id_venta están duplicados en Bronze?
WITH duplicados AS (
    SELECT id_venta, COUNT(*) AS veces
    FROM iniciacion_deportiva.bronze.ventas_raw
    GROUP BY id_venta
    HAVING COUNT(*) > 1
)
SELECT
    COUNT(*)       AS id_ventas_con_duplicados,
    SUM(veces)     AS total_registros_duplicados,
    SUM(veces - 1) AS registros_extra_eliminables
FROM duplicados;

In [ ]:
%sql
-- 5.2: Ver un ejemplo de duplicado — ¿son idénticos o cambia el estado?
-- Si cambia el estado (ej: paid → cancelled), NO son duplicados reales:
-- son actualizaciones que Silver debe procesar con MERGE
WITH dup_ids AS (
    SELECT id_venta
    FROM iniciacion_deportiva.bronze.ventas_raw
    GROUP BY id_venta
    HAVING COUNT(*) > 1
    LIMIT 3
)
SELECT v.id_venta, v.fecha, v.titulo_producto, v.estado, v.ingested_at
FROM iniciacion_deportiva.bronze.ventas_raw v
INNER JOIN dup_ids d ON v.id_venta = d.id_venta
ORDER BY v.id_venta, v.ingested_at;

In [ ]:
%sql
-- 5.3: Reporte consolidado de calidad
WITH total AS (
    SELECT COUNT(*) AS n FROM iniciacion_deportiva.bronze.ventas_raw
),
problemas AS (
    SELECT
        COUNT(CASE WHEN id_venta IS NULL OR id_venta = ''                THEN 1 END) AS sin_id_venta,
        COUNT(CASE WHEN fecha IS NULL OR fecha = ''                      THEN 1 END) AS sin_fecha,
        COUNT(CASE WHEN precio_unitario IS NULL OR precio_unitario = '' THEN 1 END) AS sin_precio,
        COUNT(CASE WHEN TRY_CAST(precio_unitario AS DOUBLE) <= 0         THEN 1 END) AS precio_invalido,
        COUNT(CASE WHEN cantidad IS NULL OR cantidad = ''                THEN 1 END) AS sin_cantidad,
        COUNT(CASE WHEN TRY_CAST(cantidad AS INT) <= 0                   THEN 1 END) AS cantidad_invalida,
        COUNT(CASE WHEN estado IS NULL OR estado = ''                    THEN 1 END) AS sin_estado,
        COUNT(CASE WHEN id_cliente IS NULL OR id_cliente = ''            THEN 1 END) AS sin_cliente
    FROM iniciacion_deportiva.bronze.ventas_raw
)
SELECT
    t.n                                            AS total_registros,
    p.sin_id_venta,
    p.sin_fecha,
    p.sin_precio,
    ROUND(p.sin_precio     * 100.0 / t.n, 2)      AS pct_sin_precio,
    p.precio_invalido,
    p.sin_cantidad,
    p.cantidad_invalida,
    p.sin_estado,
    p.sin_cliente
FROM total t, problemas p;

## Paso 6: Análisis Temporal
*¿Cómo evolucionan las ventas en el tiempo?*

In [ ]:
%sql
-- 6.1: Rango temporal del dataset
SELECT
    MIN(SUBSTRING(fecha, 1, 10))                   AS fecha_inicio,
    MAX(SUBSTRING(fecha, 1, 10))                   AS fecha_fin,
    COUNT(DISTINCT SUBSTRING(fecha, 1, 7))         AS meses_con_datos
FROM iniciacion_deportiva.bronze.ventas_raw;

In [ ]:
%sql
-- 6.2: Ventas y tasa de cancelación por mes
-- SUBSTRING(fecha, 1, 7) extrae YYYY-MM del string ISO 8601
SELECT
    SUBSTRING(fecha, 1, 7)                                             AS mes,
    COUNT(*)                                                           AS total_ordenes,
    COUNT(CASE WHEN estado = 'paid'      THEN 1 END)                   AS pagadas,
    COUNT(CASE WHEN estado = 'cancelled' THEN 1 END)                   AS canceladas,
    ROUND(
        COUNT(CASE WHEN estado = 'cancelled' THEN 1 END) * 100.0 / COUNT(*),
    2)                                                                 AS tasa_cancelacion_pct
FROM iniciacion_deportiva.bronze.ventas_raw
GROUP BY SUBSTRING(fecha, 1, 7)
ORDER BY mes;

In [ ]:
%sql
-- 6.3: Ventas por día de la semana — ¿hay días con más actividad?
SELECT
    DAYOFWEEK(TO_TIMESTAMP(fecha, "yyyy-MM-dd'T'HH:mm:ss.SSSZ"))      AS dia_num,
    CASE DAYOFWEEK(TO_TIMESTAMP(fecha, "yyyy-MM-dd'T'HH:mm:ss.SSSZ"))
        WHEN 1 THEN 'Domingo'
        WHEN 2 THEN 'Lunes'
        WHEN 3 THEN 'Martes'
        WHEN 4 THEN 'Miércoles'
        WHEN 5 THEN 'Jueves'
        WHEN 6 THEN 'Viernes'
        WHEN 7 THEN 'Sábado'
    END                                                                AS dia_semana,
    COUNT(*)                                                           AS ordenes,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2)                 AS porcentaje
FROM iniciacion_deportiva.bronze.ventas_raw
WHERE estado = 'paid'
GROUP BY 1, 2
ORDER BY 1;

In [ ]:
%sql
-- 6.4: Evolución de ingresos brutos por mes con variación MoM (Month over Month)
-- TRY_CAST: si algún string no es número devuelve NULL en vez de romper
WITH ventas_mensuales AS (
    SELECT
        SUBSTRING(fecha, 1, 7)                                         AS mes,
        COUNT(*)                                                       AS ordenes,
        ROUND(SUM(
            TRY_CAST(precio_unitario AS DOUBLE) * TRY_CAST(cantidad AS INT)
        ), 2)                                                          AS ingresos_brutos,
        ROUND(SUM(TRY_CAST(comision AS DOUBLE)), 2)                    AS total_comisiones
    FROM iniciacion_deportiva.bronze.ventas_raw
    WHERE estado = 'paid'
    GROUP BY SUBSTRING(fecha, 1, 7)
),
con_variacion AS (
    SELECT
        mes,
        ordenes,
        ingresos_brutos,
        total_comisiones,
        LAG(ingresos_brutos) OVER (ORDER BY mes)                      AS ingresos_mes_ant,
        ROUND(
            (ingresos_brutos - LAG(ingresos_brutos) OVER (ORDER BY mes))
            * 100.0 / LAG(ingresos_brutos) OVER (ORDER BY mes),
        2)                                                             AS variacion_pct
    FROM ventas_mensuales
)
SELECT
    mes,
    ordenes,
    ingresos_brutos,
    total_comisiones,
    COALESCE(CAST(variacion_pct AS STRING), 'primer mes')             AS var_vs_mes_anterior
FROM con_variacion
ORDER BY mes;

## Resumen Ejecutivo — Dashboard de Calidad
Una sola query que consolida todo lo encontrado en el EDA.

In [ ]:
%sql
WITH metricas AS (
    SELECT
        COUNT(*)                                                                        AS total_registros,
        COUNT(DISTINCT id_venta)                                                        AS ventas_unicas,
        COUNT(DISTINCT id_producto)                                                     AS productos_unicos,
        COUNT(DISTINCT id_cliente)                                                      AS clientes_unicos,
        COUNT(DISTINCT categoria_id)                                                    AS categorias,
        MIN(SUBSTRING(fecha, 1, 10))                                                    AS fecha_inicio,
        MAX(SUBSTRING(fecha, 1, 10))                                                    AS fecha_fin,
        ROUND(COUNT(CASE WHEN estado = 'paid'      THEN 1 END) * 100.0 / COUNT(*), 2)  AS pct_pagadas,
        ROUND(COUNT(CASE WHEN estado = 'cancelled' THEN 1 END) * 100.0 / COUNT(*), 2)  AS pct_canceladas,
        COUNT(*) - COUNT(DISTINCT id_venta)                                             AS duplicados_en_bronze,
        COUNT(CASE WHEN precio_unitario IS NULL OR precio_unitario = '' THEN 1 END)     AS sin_precio,
        COUNT(CASE WHEN cantidad IS NULL OR cantidad = '' THEN 1 END)                   AS sin_cantidad
    FROM iniciacion_deportiva.bronze.ventas_raw
)
SELECT
    total_registros,
    ventas_unicas,
    duplicados_en_bronze,
    productos_unicos,
    clientes_unicos,
    categorias,
    fecha_inicio,
    fecha_fin,
    pct_pagadas    || '%' AS pct_pagadas,
    pct_canceladas || '%' AS pct_canceladas,
    sin_precio,
    sin_cantidad
FROM metricas;

## Conclusiones y Decisiones para Silver

### Problemas identificados en Bronze:
1. **Duplicados** — esperados por diseño del ETL incremental (ventana 2hs, runs cada 15 min) → `MERGE ON id_venta` en Silver
2. **Todo STRING** → `TRY_CAST` a tipos correctos (DOUBLE, INT, TIMESTAMP) en Silver
3. **`fecha` como STRING ISO 8601** → parsear a TIMESTAMP con `TO_TIMESTAMP`
4. **Validaciones** → filtrar precios <= 0, cantidades <= 0

### Lo que encontramos bien:
- Sin nulos en campos clave (id_venta, fecha, precio, estado)
- `_rescued_data` en NULL → schema estable, la API no cambió estructura

### Decisión de diseño para Gold:
- Un cliente puede tener múltiples `id_venta` con la misma fecha/hora
  porque MercadoLibre devuelve una orden por producto
- En Gold agrupar por `(id_cliente, DATE_TRUNC('hour', fecha))` para reconstruir pedidos completos
- Esto permite analizar ticket promedio por pedido, no solo por ítem